In [ ]:
import numpy as np
import pandas as pd
import scipy as sp
import scipy.stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

# Сумиране по Айнщайн
Използва се при дълги алгебрични изрази за да се спести писането на суми. Улеснява и генерализира голям набор от операции от линейната алгебра. Представлява следното просто правило

#### По всички повтарящи се индекси имаме сумиране.

Например:

$ a_i B_{ij} = \sum_{i=1}^n a_i B_{ij} = c_j$

Казваме, че сме контрактирали индекса $i$. За да се случи това, размерността на двата обекта, чиито индекси контрактираме, трябва да е еднаква по въпросния индекс. Индексите, по които няма контракция (в горния пример - индекса $j$), се наричат свободни индекси. Във всяко едно равенство, където изпозлваме сумиране по Айнщайн, свободните индекси от двете страни трябва да съвпадат.

Отбеляаваме, че ние работим в Евклидова геометрия и всички индекси са долни. При не Евклидова геометрия, повтарящите се индекси трябва да са долен и горен и в сумата имаме метриката на пространството.

В python използваме функцията einsum от numpy. При нея подаваме първо индексите на обектите разделени със запетайки и индексите на крайния обек след стрелкичка. След това подаваме подред обектите. Трябва размерността на обектите да съвпада по индексите, които контрактираме. За горния пример бихме имали

`np.einsum('i,ij->j', a, B)`


## Задача 1.
Генерирайте произволни 5 мерни вектори $a$ и $b$ и произволни 5х5 матрици $A$ и $B$. Изпозлвайки сумиране по Айнщаин, пресметнете

1. Скараното произведение $a \cdot b$.

2. Произведението $A \cdot b$.

3. Матричното произведение $A \cdot B$.

4. Тензорното произведение $a \otimes b$.

5. Скаларната величина $x = a_j b_k A_{km} B_{jm}$.



## Задача 2. 
Заредете данните от файла `data/teaching_methods.csv`. Имплементирайте функция, която извършва едномерна ANOVA приемайки аргументи `(data, predictor, response)`.

$SST = \sum_{\substack{i=1\\j=1}}^{p,n} (y_{ij} - \bar y)^2 = \sum_{\substack{i=1\\j=1}}^{p,n} (y_{ij} - \bar y_i)^2 - \sum_{\substack{i=1\\j=1}}^{p,n}(\bar y_i - \bar y)^2 = SSE + SSR$

$MSE = \frac{SSE}{p(n-1)}$

$MSR = \frac{SSR}{p - 1}$

$p = P(F_{p-1,p(n-1)} > \frac{MSR}{MSE})$

In [ ]:
data = pd.read_csv('data/teaching_methods.csv')

print('Shape: ', data.shape)

In [ ]:
def anova(data, predictor, response):
    # p, n
    # group means indexed as data
    # in_group_errors
    # between_group_errors
    # SSE, SSR
    # MSE, MSR
    # F distribution


## Задача 3.
Имплементирайте функция, която извършва еднофакторна MANOVA приемайки аргументи `(data, predictor)`.

Моделът на еднофакторна ANOVA се генерализира директно към този за еднофакторна MANOVA като вземем отклика да е многомерен вектор:

Нека имаме един предиктор $X$, който може да заема $p$ на брой стойности $\{X_i\}_{i=1}^p$. Искаме да определим влиянието му над $r$ на брой отклика  $\pmb Y = (Y_1, \dots,Y_r)$. Моделът на MANOVA e:

$$
\pmb Y_{ij} = \pmb\mu + \pmb\tau_i + \pmb\varepsilon_{ij},
$$


където $\pmb\mu$ е средното на всички наблюдения, $\pmb\tau_i =\pmb\mu - \pmb\mu_i$ е отклонението на средното $\pmb\mu_i$ за $i$-тото ниво на предиктора от общото средно и $\pmb\varepsilon_{ij}$ е грешката при измерването на $\pmb Y_{ij}$.
Моделът е екивалентен на този на ANOVA, но всички величини са векторни. Приемаме, че грешките $\pmb\varepsilon_{ij}$ са некорелирани, еднакво нормално разпределени:

$$
\pmb\varepsilon_{ij} \sim \mathcal{N}(0, \sigma),\quad \mathrm{Cov}(\pmb\varepsilon_{ij}, \pmb\varepsilon_{i'j'}) = 0, \forall (i,j)\neq (i',j')
$$

Искаме да проверим дали някое от нивата на предиктора има ефект върху стойнстта на отклика. Тестваме хипотезата:

$$
H_0: \pmb\tau_i = \pmb 0,\ \forall i=1,\dots,k
$$

срещу алтернативата:

$$
H_1: \exists i: \pmb\tau_i \neq \pmb 0
$$

В едномерния случай имахме пълната сума от грешките, която беше скаларна величина. В многомерния случай имаме матрицата на пълната сума на грешките:

$$
SST = \sum_{i=1}^p \sum_{j=1}^{n_i} (\pmb Y_{ij} - \pmb\mu)\otimes(\pmb Y_{ij} - \pmb\mu)
$$

където $n_i$ е броят на измервания над $i$-тото ниво на предиктова. Отново можем да разделим тази сума на сума на отклоненията между групите и сума на отклоненията в групите. За целта разглеждаме:

$$
\begin{align*}
(\pmb Y_{ij} - \pmb\mu)\otimes(\pmb Y_{ij} - \pmb\mu )=& [(\pmb Y_{ij} - \pmb\mu_i) + (\pmb\mu_i - \pmb\mu)]\otimes[(\pmb Y_{ij} - \pmb\mu_i) + (\pmb\mu_i - \pmb\mu)] \\[8pt]
=&(\pmb Y_{ij} - \pmb\mu_i)\otimes(\pmb Y_{ij} - \pmb\mu_i) + (\pmb Y_{ij} - \pmb\mu_i)\otimes( \pmb\mu_i  - \pmb\mu)
+ (\pmb\mu_i  - \pmb\mu)\otimes(\pmb Y_{ij}- \pmb\mu_i) + (\pmb\mu_i - \pmb\mu)\otimes(\pmb\mu_i - \pmb\mu) \\[8pt]
=&(\pmb Y_{ij} - \pmb\mu_i)\otimes(\pmb Y_{ij} - \pmb\mu_i) + (\pmb Y_{ij} - \pmb\mu_i)\otimes \pmb\tau_i
+ \pmb\tau_i \otimes(\pmb Y_{ij}- \pmb\mu_i) + \pmb\tau_i \otimes \pmb\tau_i.
\end{align*}
$$

За средните два члена в сумата от грешките имаме:

$$
\sum_{i=1}^p \sum_{j=1}^{n_i} (\pmb Y_{ij} - \pmb\mu_i)\otimes \pmb\tau_i = \sum_{i=1}^p (\sum_{j=1}^{n_i}  \pmb (Y_{ij} - \pmb\mu_i))\otimes \pmb\tau_i = 0,
$$

където сме използвали линейността на външното произведение и факта, че $\pmb\mu_i$ е средното на $\pmb Y_{ij}$ за определено $i$. Следователно цялата сума става

$$
\begin{align*}
\begin{split}
SST &= \sum_{i=1}^p \sum_{j=1}^{n_i} \Big[ (\pmb Y_{ij} - \pmb\mu_i)\otimes(\pmb Y_{ij} - \pmb\mu_i) +  \pmb\tau_i \otimes \pmb\tau_i \Big]= \\
&= \sum_{i=1}^p \sum_{j=1}^{n_i} (\pmb Y_{ij} - \pmb\mu_i)\otimes(\pmb Y_{ij} - \pmb\mu_i) + \sum_{i=1}^p n_i  (\pmb\tau_i \otimes \pmb\tau_i) =\\
&= SSE + SSR = \pmb W + \pmb B.
\end{split}
\end{align*}
$$

Означили сме $SSE = \pmb W$ (sum of squares Within groups) и $SSR = \pmb B$ (sum of squared Between groups). За тестването на хипотезата $H_0$ ще изпозлваме статистиката Wilks' lambda:

$$
\Lambda*=\frac{|\pmb W|}{|\pmb W + \pmb B|}.
$$

Също като F-статистиката в едномерния случай, тя ни дава оценка за това колко за "разпръснати" резултатите в самите групи сравнено с цялостното разсейване на популацията. Ако тази статистика е достатъчно малка, можем да отхвърлим нулевата хипотеза и да заключим, че предикторът има ефект върху отклика.

Тази статистика е разпределена като F-разпределение, но параметрите на разпределението зависят от броя променливи в отклика и от броя на нивата на предиктора. За малки стойности на тези величини има таблици с точните разпределения. За голям брой променливи или нива и за голем брой измервания, Bartlett дава модификация на $\Lambda*$, която може да се използва вместо оригинала за тестване на хипотезата.

In [ ]:
from statsmodels.multivariate.manova import MANOVA

In [ ]:
model = MANOVA.from_formula('Test + Satisfaction ~ C(Method)', data=data)

In [ ]:
model.mv_test().summary()